# Giai đoạn 2: Tiền xử lý
Dự án:
Thành viên thực hiện: **Nguyễn Văn Khanh (Data Analyst)**

Sổ tay này thực hiện các công việc:
- Import thư viện và Đọc dữ liệu
- Data Type Conversions (Ép kiểu Ngày Tháng)
- Handling Missing Values (Xử lý ô trống)
- Outlier Detection and Treatment (Xử lý Ngoại lai)
- Lưu tập dữ liệu sạch ra file `data/processed/clean_prs.csv`.

In [1]:
import pandas as pd
import numpy as np
import os

RAW_DIR = "../data/raw/"
PRO_DIR = "../data/processed/"
os.makedirs(PRO_DIR, exist_ok=True)

# ==========================================
# 1. ĐỌC TOÀN BỘ 9 BẢNG DỮ LIỆU
# ==========================================
file_names = {
    'orders': 'olist_orders_dataset.csv',
    'items': 'olist_order_items_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'geo': 'olist_geolocation_dataset.csv',
    'category': 'product_category_name_translation.csv'
}
# Dùng Dictionary Comprehension để đọc 9 bảng chỉ bằng 1 dòng code
dfs = {name: pd.read_csv(os.path.join(RAW_DIR, file)) for name, file in file_names.items()}

# ==========================================
# 2. XÓA TRÙNG LẶP CHO TẤT CẢ (DUPLICATES)
# ==========================================
for name in dfs:
    dfs[name].drop_duplicates(inplace=True)

# ==========================================
# 3. LÀM SẠCH CHUYÊN SÂU CHO CÁC BẢNG QUAN TRỌNG
# ==========================================
# A. BẢNG ORDERS (Ép kiểu Datetime & Xóa dòng rỗng)
time_cols = ['order_purchase_timestamp', 'order_approved_at', 
             'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in time_cols:
    dfs['orders'][col] = pd.to_datetime(dfs['orders'][col])
# Chỉ phân tích đơn 'delivered', xóa các dòng rỗng sót lại
dfs['orders'] = dfs['orders'][dfs['orders']['order_status'] == 'delivered'].dropna(subset=['order_delivered_customer_date'])

# B. BẢNG ITEMS (Ép Capping Outliers Giá tiền bằng IQR)
Q1, Q3 = dfs['items']['price'].quantile([0.25, 0.75])
upper_bound = Q3 + 1.5 * (Q3 - Q1)
dfs['items']['price'] = np.where(dfs['items']['price'] > upper_bound, upper_bound, dfs['items']['price'])

# C. BẢNG REVIEWS (Ép Datetime & Điền khuyết bình luận)
for col in ['review_creation_date', 'review_answer_timestamp']:
    dfs['reviews'][col] = pd.to_datetime(dfs['reviews'][col])
dfs['reviews']['review_comment_message'] = dfs['reviews']['review_comment_message'].fillna('No Comment')
dfs['reviews']['review_comment_title'] = dfs['reviews']['review_comment_title'].fillna('No Title')

# D. BẢNG PRODUCTS (Điền khuyết: Tên -> 'Unknown', Kích thước -> Median)
dfs['products']['product_category_name'] = dfs['products']['product_category_name'].fillna('unknown')
num_cols = dfs['products'].select_dtypes(include=np.number).columns
dfs['products'][num_cols] = dfs['products'][num_cols].fillna(dfs['products'][num_cols].median())

# Bảng Customers, Sellers, Geo, Category mặc định đã sạch và đã được xóa trùng lặp ở Bước 2.

# ==========================================
# 4. XUẤT 9 BẢNG SẠCH RA THƯ MỤC PROCESSED
# ==========================================
for name, df in dfs.items():
    clean_filename = f"clean_{file_names[name]}"
    df.to_csv(os.path.join(PRO_DIR, clean_filename), index=False)

print("✅ Đã làm sạch toàn bộ 9 bảng và lưu vào thư mục data/processed/")

✅ Đã làm sạch toàn bộ 9 bảng và lưu vào thư mục data/processed/
